# Field Transformations Quickstart

`FieldTransformation(source, expr)` rewrites compiled interaction terms — the FeynPy
analogue of FeynRules `Definitions`.

**Workflow:** define fields → build `Model` → compile `lagrangian()` →
`transform_fields(...)` → read `.feynman_rule(...)`.

**Expression syntax** (one rule per line):

| Pattern | Example |
|---------|---------|
| Linear mixing | `FieldTransformation(B, -sw * Z + cw * A)` |
| Vacuum shift | `FieldTransformation(Phi, vev * c + c * H, components={0: 2})` |
| Chiral projector | `FieldTransformation(LL, ProjM * l, components={1: 2})` |

This notebook shows the boson-mixing case and compares Feynman rules before vs after.


In [1]:
from __future__ import annotations
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import S

from feynpy import (
    CovD,
    Field,
    FieldTransformation,
    GaugeGroup,
    LORENTZ_INDEX,
    Model,
    WEAK_FUND_INDEX,
)


In [2]:
def safe_rule(L, *fields):
    try:
        return L.feynman_rule(*fields)
    except ValueError:
        return 0

## 1) Define fields and source-basis model

We use one charged scalar doublet `H` and one neutral gauge field `B`.
Then we define two physical vector fields `A` and `Z` that will be introduced
only through transformations.


In [3]:
mu = S("mu")
g1 = S("g1")
sw = S("sw")
cw = S("cw")
yH = S("yH")

B = Field(
    "B",
    spin=1,
    self_conjugate=True,
    symbol=S("B0"),
    indices=(LORENTZ_INDEX,),
)

A = Field(
    "A",
    spin=1,
    self_conjugate=True,
    symbol=S("A0"),
    indices=(LORENTZ_INDEX,),
)

Z = Field(
    "Z",
    spin=1,
    self_conjugate=True,
    symbol=S("Z0"),
    indices=(LORENTZ_INDEX,),
)

H = Field(
    "H",
    spin=0,
    self_conjugate=False,
    symbol=S("H0"),
    conjugate_symbol=S("Hdag0"),
    indices=(WEAK_FUND_INDEX,),
    quantum_numbers={"Y": yH},
)

U1Y = GaugeGroup(
    name="U1Y",
    abelian=True,
    coupling=g1,
    gauge_boson="B",
    charge="Y",
)

source_model = Model(
    name="transform-quickstart",
    gauge_groups=(U1Y,),
    fields=(H, B, A, Z),
    lagrangian_decl=CovD(H.bar, mu) * CovD(H, mu),
)


## 2) Before transformations

In the source basis, vertices exist with `B`, but not with `A`/`Z`.


In [4]:
L_source = source_model.lagrangian()

print("Source basis:")
print("  Γ(Hdag, H, B):", safe_rule(L_source, H.bar, H, B))
print("  Γ(Hdag, H, A):", safe_rule(L_source, H.bar, H, A))
print("  Γ(Hdag, H, Z):", safe_rule(L_source, H.bar, H, Z))


Source basis:
  Γ(Hdag, H, B): -1𝑖*g1*yH*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu3)+1𝑖*g1*yH*g(cof(2, w1),cof(2, w2))*pcomp(q2,mu3)
  Γ(Hdag, H, A): 0
  Γ(Hdag, H, Z): 0


## 3) Apply the transformation

One rule, FeynRules-style:

```python
FieldTransformation(B, -sw * Z + cw * A)
```

`transform_fields` rewrites every `B` occurrence inside the compiled Lagrangian.
No Python reassignment — `.feynman_rule(...)` automatically sees `A` and `Z` instead of `B`.


In [5]:
rules = (FieldTransformation(B, -sw * Z + cw * A),)

L_physical = source_model.transform_fields(*rules, repeat=True)

print("After transformation:")
print("  Γ(Hdag, H, B):", safe_rule(L_physical, H.bar, H, B))
print("  Γ(Hdag, H, A):", safe_rule(L_physical, H.bar, H, A))
print("  Γ(Hdag, H, Z):", safe_rule(L_physical, H.bar, H, Z))


After transformation:
  Γ(Hdag, H, B): 0
  Γ(Hdag, H, A): -1𝑖*g1*cw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu3)+1𝑖*g1*cw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q2,mu3)
  Γ(Hdag, H, Z): 1𝑖*g1*sw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu3)-1𝑖*g1*sw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q2,mu3)


## 4) Summary

- `B` vertices vanish; `A` and `Z` vertices appear with coefficients `cw` and `sw`.
- The same `FieldTransformation(source, expr)` syntax also handles vacuum shifts
  (`vev * c + c * H`) and matrix factors (`ProjM * l`, `rotation(CKM, CKMDag) * ProjM * dq`).
- See `standard_model_feynpy.ipynb` section 7 for the full SM `Definitions` block.
